# Euro Coin Detector - Methods and Logic Report

This notebook documents the **current final architecture** and explains the full logic from input image to predicted EUR value.

Goal of this notebook:
- make the codebase understandable for presentation and defense,
- show which module is responsible for each stage,
- provide runnable code blocks to inspect and evaluate the pipeline.

## Contents

1. Environment setup
2. Project structure and modules
3. Dataset and annotation overview
4. Detection configuration snapshot
5. Pipeline architecture (logic flow)
6. API map and source-code inspection
7. End-to-end demo on one image
8. Mini benchmark on a subset
9. Final takeaways

In [ ]:
# Setup imports and path resolution so notebook can run from project root or parent folder
from pathlib import Path
import sys
import inspect
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "Image_projet_money" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Image_projet_money"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import cv2
    CV2_AVAILABLE = True
except Exception as e:
    CV2_AVAILABLE = False
    print("OpenCV not available in this environment:", e)

from src.config import DetectionConfig, RuntimeConfig
from src.dataset import DatasetRepository
from src.io_utils import ImagePathResolver
from src.processor import CoinProcessor
from src.processor_circles import CircleDetector
from src.processor_color import CoinColorClassifier
from src.processor_scale import ScaleValueClassifier
from src.coin_metadata import COIN_DIAMETER_MM, COLOR_TO_DENOMS

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CV2_AVAILABLE:", CV2_AVAILABLE)

## 1) Project Structure and Module Roles

The current codebase is separated by responsibility:
- `src/processor.py`: pipeline orchestration,
- `src/processor_circles.py`: circle detection and geometric filtering,
- `src/processor_color.py`: LAB/HSV center-ring color logic,
- `src/processor_scale.py`: global scale fitting and denomination assignment,
- `src/runner.py` + `src/visualization.py`: batch run and UI tools.

In [ ]:
# Print a compact tree of main project files
interesting = {
    "main.py",
    "README.md",
    "PIPELINE.md",
    "src/config.py",
    "src/models.py",
    "src/io_utils.py",
    "src/dataset.py",
    "src/coin_metadata.py",
    "src/processor.py",
    "src/processor_circles.py",
    "src/processor_color.py",
    "src/processor_scale.py",
    "src/runner.py",
    "src/visualization.py",
}

for p in sorted(PROJECT_ROOT.rglob("*.py")):
    rel = p.relative_to(PROJECT_ROOT).as_posix()
    if rel in interesting or rel.startswith("report/"):
        print(rel)

## 2) Dataset and Ground Truth Overview

We use `DatasetRepository.DATA_ROWS` as the reference table:
- `image`: filename,
- `pieces`: expected coin count,
- `value_eur`: expected total value (when available),
- `group`: subfolder key.

In [ ]:
repo = DatasetRepository()
df_gt = repo.to_dataframe()

print("Total annotated images:", len(df_gt))
print("Groups:", sorted(df_gt["group"].unique()))
print()
print(df_gt.head(10))

summary = (
    df_gt.groupby("group")
    .agg(num_images=("image", "count"), avg_pieces=("pieces", "mean"), avg_value=("value_eur", "mean"))
    .reset_index()
)
summary

## 3) Detection Configuration Snapshot

These are the default values used by `CoinProcessor.execute()`.

In [ ]:
cfg = DetectionConfig()
cfg_df = pd.DataFrame(sorted(cfg.__dict__.items()), columns=["parameter", "value"])
cfg_df

## 4) Pipeline Logic (High-Level)

```text
Input BGR image
  -> resize to target width
  -> grayscale normalize (+ optional inversion for dark images)
  -> median blur
  -> circle ensemble detection (strict Hough + loose Hough + contour backup)
  -> per-coin color features (center/ring LAB+HSV)
  -> color-group scores and candidate denominations
  -> global px/mm scale fit on all detected coins
  -> final denomination labels + total EUR
```

Design rationale:
- Geometry first to stabilize detection,
- Color used as a **soft prior** (not hard rule),
- One global scale enforces consistency across all coins in one image.

## 5) API Map and Method Inventory

The table below shows which main methods exist in each major class.

In [ ]:
def public_methods(cls):
    names = []
    for name, fn in inspect.getmembers(cls, inspect.isfunction):
        if name.startswith("__"):
            continue
        names.append(name)
    return names

api_rows = []
for cls in [CoinProcessor, CircleDetector, CoinColorClassifier, ScaleValueClassifier]:
    for m in public_methods(cls):
        api_rows.append({"class": cls.__name__, "method": m})

api_df = pd.DataFrame(api_rows).sort_values(["class", "method"]).reset_index(drop=True)
api_df

## 6) Key Source Blocks (Directly from Code)

These code blocks are extracted from current modules so the report reflects the real implementation.

In [ ]:
def show_source(fn, max_lines=80):
    src = inspect.getsource(fn).splitlines()
    clipped = src[:max_lines]
    print("".join(clipped))
    if len(src) > max_lines:
        print("... (truncated)")

print("[CoinProcessor.detect_with_params]")
show_source(CoinProcessor.detect_with_params, max_lines=110)

In [ ]:
print("[CircleDetector.detect_ensemble]")
show_source(CircleDetector.detect_ensemble, max_lines=120)

print("[CoinColorClassifier.extract_coin_features]")
show_source(CoinColorClassifier.extract_coin_features, max_lines=120)

print("[ScaleValueClassifier.classify]")
show_source(ScaleValueClassifier.classify, max_lines=120)

## 7) End-to-End Demo on One Image

This section runs the final pipeline and displays all generated debug stages.

In [ ]:
def show_bgr(img, title="", figsize=(5, 5)):
    if img is None:
        print("None image")
        return
    plt.figure(figsize=figsize)
    if CV2_AVAILABLE and img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

resolver = ImagePathResolver(RuntimeConfig().IMAGE_DIRECTORY)
processor = CoinProcessor(DetectionConfig())

sample_row = df_gt.iloc[0]
sample_path = resolver.resolve(sample_row["image"], sample_row["group"])
print("Sample:", sample_row["group"], sample_row["image"])
print("Path:", sample_path)

if CV2_AVAILABLE and sample_path:
    img = cv2.imread(sample_path)
    result = processor.execute(img, filename=sample_row["image"])
    print("Detected coins:", result.coin_count)
    print("Predicted total EUR:", round(result.estimated_value_eur, 2))
    print("Labeled coins:", result.labeled_coin_count)

    for i, step in enumerate(result.steps):
        cmap = "gray" if step.cmap == "gray" else None
        plt.figure(figsize=(5, 4))
        if step.cmap == "rgb":
            plt.imshow(cv2.cvtColor(step.image, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(step.image, cmap=cmap)
        plt.title(f"{i+1}. {step.name}")
        plt.axis("off")
        plt.show()
else:
    print("Skipping demo: OpenCV or sample image unavailable.")

## 8) Per-Coin Diagnostic View

This prints the per-coin tags, radii, color labels, and candidate denomination sets.

In [ ]:
if CV2_AVAILABLE and sample_path:
    img = cv2.imread(sample_path)
    result = processor.execute(img, filename=sample_row["image"])

    print("coin_tags:", result.coin_tags)
    print("coin_radii:", [round(r, 2) for r in result.coin_radii])
    print("coin_color_labels:", result.coin_color_labels)
    print("coin_candidate_denoms:", result.coin_candidate_denoms)
    print("coin_labels:", result.coin_labels)

    if result.radius_ratio_matrix:
        ratio_df = pd.DataFrame(result.radius_ratio_matrix, index=result.coin_tags, columns=result.coin_tags)
        display(ratio_df)
else:
    print("Skipping diagnostics: OpenCV or sample image unavailable.")

## 9) Mini Benchmark (Subset)

A short run to quickly estimate count and value behavior on a subset.

In [ ]:
def benchmark_subset(max_images=20):
    rows = []
    sub = df_gt.head(max_images)

    for _, row in sub.iterrows():
        path = resolver.resolve(row["image"], row["group"])
        if not path:
            continue
        img = cv2.imread(path) if CV2_AVAILABLE else None
        if img is None:
            continue

        res = processor.execute(img, filename=row["image"])
        pred_count = int(res.coin_count)
        true_count = int(row["pieces"])
        pred_value = float(res.estimated_value_eur)
        true_value = float(row["value_eur"]) if pd.notna(row["value_eur"]) else np.nan

        rows.append(
            {
                "image": row["image"],
                "group": row["group"],
                "pred_count": pred_count,
                "true_count": true_count,
                "count_abs_err": abs(pred_count - true_count),
                "pred_value": pred_value,
                "true_value": true_value,
                "value_abs_err": abs(pred_value - true_value) if pd.notna(true_value) else np.nan,
            }
        )

    out = pd.DataFrame(rows)
    if out.empty:
        return out, {}

    metrics = {
        "images": len(out),
        "count_mae": float(out["count_abs_err"].mean()),
        "count_exact_rate_%": float((out["count_abs_err"] == 0).mean() * 100.0),
        "value_mae": float(out["value_abs_err"].dropna().mean()) if out["value_abs_err"].notna().any() else np.nan,
    }
    return out, metrics

if CV2_AVAILABLE:
    bench_df, bench_metrics = benchmark_subset(max_images=20)
    display(bench_df.head(10))
    print("Metrics:", bench_metrics)
else:
    print("Skipping benchmark: OpenCV not available.")

## 10) Final Notes

This notebook gives a complete technical map of the current system:
- where each algorithm lives,
- how the logic flows,
- and how to run practical checks.

For progression history (what we tried before final Hough + center-ring HSV/LAB), see `progress_hough_hsv_report.ipynb`.